# Ablation v2 — LDM vs DiffusionSR vs Flow-Matching
Interactive comparison notebook.  
Axes: **model** × **encoder** × **field** × **normalization** × **QSS frame**

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

# Adjust if running locally vs on TRACE
RUNS_ROOT   = Path('/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_v2')
DATA_ROOT   = Path('/trace/group/forgelab/ngng/multifield/data_fields')
WANDB_PROJECT = 'Flow3D_SuperResolution'
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', '')
N_QSS = 5    # last N test frames = quasi-steady state window
N_SAMPLES = 5  # ensemble samples per frame

# Add repo root to path
REPO = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path.cwd().parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json, warnings
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import torch
import wandb
import ipywidgets as widgets
from IPython.display import display, clear_output

from diffusionsr.datasets.flipped_dataset import FlippedDataset

warnings.filterwarnings('ignore')

In [ ]:
# ── Run manifest ──────────────────────────────────────────────────────────────
# Each run is identified by a (model, enc, field, norm) tuple.
# run_dirs maps that key to the Path of its results directory.

MODELS = ['diffusion', 'fm', 'ldm']
ENCS   = ['enc', 'noenc']
FIELDS = ['temp', 'multi']
NORMS  = ['std', 'gstd']

FIELD_NAMES = {'temp': ['temperature'], 'multi': ['temperature', 'sdfliqlabel']}
NORM_KEY    = {'std': 'standardize',    'gstd': 'global_standardize'}
MODEL_LABEL = {'diffusion': 'DiffusionSR', 'fm': 'Flow-Matching', 'ldm': 'LDM'}

run_dirs = {}
for m in MODELS:
    for e in ENCS:
        for f in FIELDS:
            for n in NORMS:
                d = RUNS_ROOT / f'{m}_{e}_{f}_{n}'
                if d.exists():
                    run_dirs[(m, e, f, n)] = d

print(f'{len(run_dirs)}/24 run directories found')
for k, v in run_dirs.items():
    ckpt = (v / 'bestmodel_saved.pth').exists() or (v / 'ckpt.pth').exists()
    print(f"  {k} — {'✓ ckpt' if ckpt else '✗ no ckpt'}")

## 1 — Training time

In [ ]:
# Load training_time.txt from each run directory (written by train_diffusion.py)
train_times = {}
for key, d in run_dirs.items():
    tt_file = d / 'training_time.txt'
    vae_file = d / 'vae_training_time.txt'
    t = 0.0
    if tt_file.exists():
        t += float(np.loadtxt(tt_file))
    if vae_file.exists():
        t += float(np.loadtxt(vae_file))  # LDM: add VAE pretraining time
    if t > 0:
        train_times[key] = t / 3600  # seconds → hours

print(f'Training times loaded for {len(train_times)} runs')

In [ ]:
# Bar chart: training time per model architecture
if train_times:
    rows = []
    for (m, e, f, n), t in train_times.items():
        rows.append({'Model': MODEL_LABEL[m], 'Encoder': e, 'Field': f, 'Norm': n, 'Time (h)': t})
    df_t = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for ax, field_key, field_label in zip(axes, ['temp', 'multi'], ['Single field (temp)', 'Multifield']):
        sub = df_t[df_t['Field'] == field_key]
        x = np.arange(len(MODELS))
        width = 0.18
        for i, (e, n) in enumerate([('enc','std'), ('enc','gstd'), ('noenc','std'), ('noenc','gstd')]):
            vals = [sub[(sub['Model']==MODEL_LABEL[m]) & (sub['Encoder']==e) & (sub['Norm']==n)]['Time (h)'].values
                    for m in MODELS]
            vals = [v[0] if len(v) else 0 for v in vals]
            label = f"{e}/{n}"
            ax.bar(x + i*width, vals, width, label=label)
        ax.set_xticks(x + 1.5*width)
        ax.set_xticklabels([MODEL_LABEL[m] for m in MODELS])
        ax.set_ylabel('Wall-clock hours')
        ax.set_title(field_label)
        ax.legend(fontsize=7)
    plt.suptitle('Training time by model architecture', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No training_time.txt files found yet — run training first.')

## 2 — Metrics table

In [ ]:
# Load W&B summary metrics for all runs via the W&B API.
# Falls back to a local results.json if the W&B run is not reachable.

api = wandb.Api()
wb_metrics = {}

for key, d in run_dirs.items():
    # Try local results.json first
    local_json = d / 'results.json'
    if local_json.exists():
        with open(local_json) as fh:
            wb_metrics[key] = json.load(fh)
        continue

    # Fall back to W&B API via wandb_run_id.txt
    id_file = d / 'wandb_run_id.txt'
    if not id_file.exists():
        continue
    run_id = id_file.read_text().strip()
    try:
        run = api.run(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{run_id}')
        wb_metrics[key] = {
            'psnr':    run.summary.get('psnr'),
            'ssim':    run.summary.get('ssim'),
            'mp_mae':  run.summary.get('mp_mae'),
            'train_t': run.summary.get('training_time_s'),
        }
    except Exception as exc:
        print(f'W&B fetch failed for {key}: {exc}')

print(f'Metrics loaded for {len(wb_metrics)} runs')

In [ ]:
# Styled metrics table: rows = model+enc+field, columns = std / gstd
def make_metrics_table(field_key):
    rows = []
    for m in MODELS:
        for e in ENCS:
            row = {'Model': MODEL_LABEL[m], 'Enc': e}
            for n in NORMS:
                k = (m, e, field_key, n)
                d = wb_metrics.get(k, {})
                row[f'PSNR ({n})']   = d.get('psnr')
                row[f'SSIM ({n})']   = d.get('ssim')
                row[f'MP-MAE ({n})'] = d.get('mp_mae')
            rows.append(row)
    return pd.DataFrame(rows).set_index(['Model', 'Enc'])

if wb_metrics:
    print('=== Single field (temperature) ===')
    display(make_metrics_table('temp').style.background_gradient(cmap='RdYlGn', axis=0))
    print('\n=== Multifield ===')
    display(make_metrics_table('multi').style.background_gradient(cmap='RdYlGn', axis=0))
else:
    print('No metrics available yet.')

## 3 — QSS overlay (interactive)

Use the controls below to select a variant. The slider scrubs through the last 5 test-set frames (quasi-steady state region). Each panel shows GT temperature | predicted temperature | absolute error, for all three models side-by-side.

In [ ]:
# ── Load test datasets (one per field × normalize combination) ─────────────
# Datasets are shared across models since data loading is independent of model.

test_datasets = {}
for f in FIELDS:
    for n in NORMS:
        try:
            ds = FlippedDataset(
                split='test',
                downscale_method='direct',
                root_folder=str(DATA_ROOT),
                normalize=NORM_KEY[n],
                n_steps=5,
                field_names=FIELD_NAMES[f],
                p_flip=0.0,
            )
            test_datasets[(f, n)] = ds
            print(f'Loaded test_ds ({f}, {n}): {len(ds)} samples, img_shape={ds.img_shape}')
        except Exception as exc:
            print(f'Dataset ({f}, {n}) failed: {exc}')

In [ ]:
# ── Model loader cache ──────────────────────────────────────────────────────
# Models are loaded lazily and cached to avoid reloading on every widget interaction.

_model_cache = {}

def _load_model(key):
    """Load and cache the model for (model_type, enc, field, norm)."""
    if key in _model_cache:
        return _model_cache[key]
    m, e, f, n = key
    if key not in run_dirs:
        return None
    d = run_dirs[key]
    ds = test_datasets.get((f, n))
    if ds is None:
        return None

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    from diffusionsr.datasets.flipped_dataset import FlippedDataset as _FD
    train_ds = _FD(split='train', downscale_method='direct', root_folder=str(DATA_ROOT),
                   normalize=NORM_KEY[n], n_steps=5, field_names=FIELD_NAMES[f], p_flip=0.0)
    dev_ds   = _FD(split='dev',   downscale_method='direct', root_folder=str(DATA_ROOT),
                   normalize=NORM_KEY[n], n_steps=5, field_names=FIELD_NAMES[f], p_flip=0.0)

    enc_dir = run_dirs.get((m, e, f, n))

    try:
        if m == 'ldm':
            from diffusionsr.runners.train_ldm import LDMModel
            # Peek for vae_results_dir — stored in configuration.yml if present
            cfg_path = d / 'configuration.yml'
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text()) if cfg_path.exists() else {}
            vae_dir = cfg.get('vae_results_dir', str(d))
            model = LDMModel(vae_folder=vae_dir, results_folder=str(d),
                             lr_encoder_folder=str(enc_dir) if e == 'enc' else None,
                             train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=ds,
                             encoding=(e == 'enc'), device=device)
        elif m == 'fm':
            from diffusionsr.runners.train_flow_matching import FlowMatchingModel
            model = FlowMatchingModel(results_folder=str(d),
                                      lr_encoder_folder=str(enc_dir) if e == 'enc' else None,
                                      train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=ds,
                                      encoding=(e == 'enc'), device=device)
        else:
            from diffusionsr.runners.train_diffusion import DiffusionModel
            model = DiffusionModel(results_folder=str(d),
                                   lr_encoder_folder=str(enc_dir) if e == 'enc' else None,
                                   train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=ds,
                                   encoding=(e == 'enc'), device=device)

        best_path = d / 'bestmodel_saved.pth'
        ckpt_path = d / 'ckpt.pth'
        load_path = best_path if best_path.exists() else ckpt_path
        if load_path.exists():
            ckpt = torch.load(load_path, map_location=device)
            model.model.load_state_dict(ckpt[0])
        model.model.eval()
        _model_cache[key] = model
        return model
    except Exception as exc:
        print(f'Model load failed {key}: {exc}')
        return None

In [ ]:
# ── Prediction cache ──────────────────────────────────────────────────────────
# pred_cache[(m,e,f,n)] = (pred_frames, gt_frames, ulr_frames)
#   shapes: (N_QSS, C, H, W) in physical units
#
# Fast path: load from precomputed preview/qss_samples.npz if it exists.
# Run scripts/precompute_ablation_v2_preview.py on TRACE to populate these.

_pred_cache = {}

def _get_preds(key):
    if key in _pred_cache:
        return _pred_cache[key]
    m, e, f, n = key
    if key not in run_dirs:
        return None

    # ── Fast path: precomputed cache ──────────────────────────────────────────
    npz_path = run_dirs[key] / 'preview' / 'qss_samples.npz'
    if npz_path.exists():
        data = np.load(str(npz_path))
        result = (data['preds'], data['gts'], data['ulrs'])
        _pred_cache[key] = result
        return result

    # ── Slow path: live inference ──────────────────────────────────────────────
    ds = test_datasets.get((f, n))
    model = _load_model(key)
    if ds is None or model is None:
        return None

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    qss_indices = list(range(len(ds) - N_QSS, len(ds)))
    preds, gts, ulrs = [], [], []

    for idx in qss_indices:
        res, hr, lr, ulr = ds[idx]
        n_ch = len(ds.field_names)
        hr_last  = hr[-n_ch:].unsqueeze(0).to(device).float()
        ulr_last = ulr[-n_ch:].unsqueeze(0).to(device).float()
        lr_t     = lr.unsqueeze(0).to(device).float()
        ulr_full = ulr.unsqueeze(0).to(device).float()

        with torch.no_grad():
            sampler = 'euler' if m == 'fm' else 'DDIM'
            samples = model.batch_sample(ulr_full, lr_t, sampler=sampler, skip=20)
            pred = samples.mean(0, keepdim=True) if samples.ndim == 5 else samples

        pred_phys = ds.unscale_data(pred.cpu().numpy()[0],   input_type='hr')
        gt_phys   = ds.unscale_data(hr_last.cpu().numpy()[0], input_type='hr')
        ulr_phys  = ds.unscale_data(ulr_last.cpu().numpy()[0], input_type='upscaled_lr')

        preds.append(pred_phys)
        gts.append(gt_phys)
        ulrs.append(ulr_phys)

    result = (np.stack(preds), np.stack(gts), np.stack(ulrs))
    _pred_cache[key] = result
    return result

In [ ]:
# ── Interactive overlay widget ─────────────────────────────────────────────────

enc_dd    = widgets.Dropdown(options=[('With encoder', 'enc'), ('No encoder', 'noenc')],
                              description='Encoder:', style={'description_width': '80px'})
field_dd  = widgets.Dropdown(options=[('Temperature', 'temp'), ('Multifield', 'multi')],
                              description='Fields:', style={'description_width': '80px'})
norm_dd   = widgets.Dropdown(
    options=[('Pixel-wise standardize', 'std'), ('Global standardize', 'gstd')],
    description='Normalize:', style={'description_width': '80px'})
frame_sl  = widgets.IntSlider(min=0, max=N_QSS-1, value=0, description='QSS frame:',
                               style={'description_width': '80px'}, layout=widgets.Layout(width='350px'))
status_lbl = widgets.Label(value='Select options and press Run.')
run_btn   = widgets.Button(description='Run / Refresh', button_style='primary')

overlay_out = widgets.Output()

CMAP_T = 'jet'
VMIN_T, VMAX_T = 293, 5000

def _plot_overlay(change=None):
    enc_v   = enc_dd.value
    field_v = field_dd.value
    norm_v  = norm_dd.value
    frame_v = frame_sl.value

    with overlay_out:
        clear_output(wait=True)
        status_lbl.value = 'Loading predictions…'

        all_preds = {}
        for m in MODELS:
            key = (m, enc_v, field_v, norm_v)
            res = _get_preds(key)
            if res is not None:
                all_preds[m] = res

        if not all_preds:
            status_lbl.value = 'No predictions available for this combination.'
            return

        # Use GT from first available model (all share the same dataset)
        first_key = list(all_preds.keys())[0]
        _, gt_frames, ulr_frames = all_preds[first_key]
        gt  = gt_frames[frame_v, 0]   # temperature channel, shape (H, W)
        ulr = ulr_frames[frame_v, 0]

        n_cols = 1 + len(all_preds)   # GT + one per model
        fig, axes = plt.subplots(2, n_cols, figsize=(4*n_cols, 7),
                                  gridspec_kw={'hspace': 0.35, 'wspace': 0.12})
        kw = dict(cmap=CMAP_T, vmin=VMIN_T, vmax=VMAX_T, origin='lower', aspect='auto')
        ekw = dict(cmap='hot', vmin=0, vmax=500, origin='lower', aspect='auto')

        # GT column
        im0 = axes[0, 0].imshow(gt.T, **kw)
        axes[0, 0].set_title('Ground Truth', fontsize=10)
        axes[1, 0].imshow(ulr.T, **kw)
        axes[1, 0].set_title('Upscaled LR (input)', fontsize=10)
        plt.colorbar(im0, ax=axes[:, 0], fraction=0.04, label='T [K]')

        for col, m in enumerate(MODELS, start=1):
            if m not in all_preds:
                axes[0, col].axis('off')
                axes[1, col].axis('off')
                continue
            pred_frames, _, _ = all_preds[m]
            pred = pred_frames[frame_v, 0]
            err  = np.abs(pred - gt)

            axes[0, col].imshow(pred.T, **kw)
            axes[0, col].set_title(MODEL_LABEL[m], fontsize=10)
            im_e = axes[1, col].imshow(err.T, **ekw)
            axes[1, col].set_title(f'|Error| — {MODEL_LABEL[m]}', fontsize=10)

        plt.colorbar(im_e, ax=axes[1, 1:], fraction=0.04, label='|ΔT| [K]')

        norm_label = dict(std='pixel-wise std', gstd='global std')[norm_v]
        field_label = dict(temp='temperature', multi='multifield')[field_v]
        enc_label   = dict(enc='encoder', noenc='no encoder')[enc_v]
        fig.suptitle(
            f'QSS frame {frame_v+1}/{N_QSS}  |  {field_label}  |  {enc_label}  |  {norm_label}',
            fontsize=11)
        plt.show()
        status_lbl.value = 'Done.'

run_btn.on_click(_plot_overlay)
frame_sl.observe(_plot_overlay, names='value')

controls = widgets.HBox([enc_dd, field_dd, norm_dd, frame_sl, run_btn])
display(controls, status_lbl, overlay_out)
_plot_overlay()  # initial render

## 4 — Ensemble standard deviation maps

In [ ]:
# ── Ensemble std cache ────────────────────────────────────────────────────────
# Draws N_SAMPLES per frame and computes per-pixel std.
# Fast path: loads stds from preview/qss_samples.npz if precomputed.

_std_cache = {}

def _get_std(key):
    if key in _std_cache:
        return _std_cache[key]
    m, e, f, n = key
    if key not in run_dirs:
        return None

    # ── Fast path: precomputed cache ──────────────────────────────────────────
    npz_path = run_dirs[key] / 'preview' / 'qss_samples.npz'
    if npz_path.exists():
        data = np.load(str(npz_path))
        result = data['stds']  # (N_QSS, H, W)
        _std_cache[key] = result
        return result

    # ── Slow path: live inference ──────────────────────────────────────────────
    ds = test_datasets.get((f, n))
    model = _load_model(key)
    if ds is None or model is None:
        return None

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    qss_indices = list(range(len(ds) - N_QSS, len(ds)))
    stds = []
    for idx in qss_indices:
        _, hr, lr, ulr = ds[idx]
        n_ch = len(ds.field_names)
        lr_t   = lr.unsqueeze(0).to(device).float()
        ulr_t  = ulr.unsqueeze(0).to(device).float()
        samples_list = []
        with torch.no_grad():
            sampler = 'euler' if m == 'fm' else 'DDIM'
            for _ in range(N_SAMPLES):
                s = model.batch_sample(ulr_t, lr_t, sampler=sampler, skip=20)
                s_phys = ds.unscale_data(s.cpu().numpy()[0], input_type='hr')
                samples_list.append(s_phys[0])
        stds.append(np.std(np.stack(samples_list), axis=0))

    result = np.stack(stds)
    _std_cache[key] = result
    return result

In [ ]:
# ── Ensemble std interactive widget ───────────────────────────────────────────

std_enc_dd   = widgets.Dropdown(options=[('With encoder', 'enc'), ('No encoder', 'noenc')],
                                 description='Encoder:', style={'description_width': '80px'})
std_field_dd = widgets.Dropdown(options=[('Temperature', 'temp'), ('Multifield', 'multi')],
                                 description='Fields:', style={'description_width': '80px'})
std_norm_dd  = widgets.Dropdown(
    options=[('Pixel-wise standardize', 'std'), ('Global standardize', 'gstd')],
    description='Normalize:', style={'description_width': '80px'})
std_frame_sl = widgets.IntSlider(min=0, max=N_QSS-1, value=0, description='QSS frame:',
                                  style={'description_width': '80px'}, layout=widgets.Layout(width='350px'))
std_run_btn  = widgets.Button(description='Run / Refresh', button_style='primary')
std_out      = widgets.Output()

def _plot_std(change=None):
    enc_v   = std_enc_dd.value
    field_v = std_field_dd.value
    norm_v  = std_norm_dd.value
    frame_v = std_frame_sl.value

    with std_out:
        clear_output(wait=True)
        all_stds = {}
        for m in MODELS:
            s = _get_std((m, enc_v, field_v, norm_v))
            if s is not None:
                all_stds[m] = s

        if not all_stds:
            print('No std maps available.')
            return

        fig, axes = plt.subplots(1, len(all_stds), figsize=(5*len(all_stds), 4))
        axes = np.atleast_1d(axes)
        vmax = max(s[frame_v].max() for s in all_stds.values())
        for ax, (m, s) in zip(axes, all_stds.items()):
            im = ax.imshow(s[frame_v].T, cmap='hot', vmin=0, vmax=vmax,
                           origin='lower', aspect='auto')
            ax.set_title(MODEL_LABEL[m], fontsize=10)
            plt.colorbar(im, ax=ax, fraction=0.05, label='σ [K]')

        norm_label = dict(std='pixel-wise std', gstd='global std')[norm_v]
        field_label = dict(temp='temperature', multi='multifield')[field_v]
        fig.suptitle(f'Ensemble σ  |  QSS frame {frame_v+1}/{N_QSS}  |  {field_label}  |  {norm_label}', fontsize=11)
        plt.tight_layout()
        plt.show()

std_run_btn.on_click(_plot_std)
std_frame_sl.observe(_plot_std, names='value')

std_controls = widgets.HBox([std_enc_dd, std_field_dd, std_norm_dd, std_frame_sl, std_run_btn])
display(std_controls, std_out)
_plot_std()